# Controls, batch mode: runs a whole list of (architecture, seed) combinations one after another

For every combination it does everything: trains (50 epochs, best APTOS validation QWK), tests on APTOS (once), IDRiD (once) and DDR deduplicated (once), and saves the files under the same names as the manual notebook.

**You only run the cells top to bottom, then wait.** If the session is cut off, run the same cells again with the same settings: finished runs are skipped, and an unfinished run resumes from its last saved epoch.

1. Fill the three paths in cell 1.
2. In cell 2 leave `SMOKE = True` for the first pass (2 epochs, tag `batchsmoke`, one combination). Check there are no errors.
3. Set `SMOKE = False` in cell 2, run cell 2 again, then run the remaining cells again (from cell 2 down) for the real batch.

## 1. Setup (paths, unzip)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile

APTOS_ZIP = "/content/drive/MyDrive/your folder.zip"
IDRID_ZIP = "/content/drive/MyDrive/your folder.zip"        # <-- CHANGE
DDR_ZIP   = "/content/drive/MyDrive/your folder.zip"          # <-- CHANGE

os.system("pip install -q timm")
if not os.path.exists("/content/APTOS"):
    zipfile.ZipFile(APTOS_ZIP).extractall("/content/APTOS")
if not os.path.exists("/content/idrid/IDRID"):
    os.system(f'unzip -q "{IDRID_ZIP}" -d /content/idrid')
if not os.path.exists("/content/ddr/DR_grading.csv"):
    os.system(f'unzip -q "{DDR_ZIP}" -d /content/ddr')
print(os.listdir("/content/APTOS"), os.listdir("/content/idrid"), os.listdir("/content/ddr"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['test.csv', 'train_1.csv', 'test_images', 'valid.csv', 'train_images', 'val_images'] ['IDRID'] ['DR_grading.csv', 'DR_grading']


## 2. Imports and the list of runs

In [ ]:
import random, hashlib, time
import numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import timm, cv2
from concurrent.futures import ThreadPoolExecutor
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score, roc_auc_score,
                             classification_report, confusion_matrix)

BASE_PATH = "/content/drive/MyDrive/APTOS-Finalkefinal(modified-NEW1)"
os.makedirs(BASE_PATH + "/checkpoints", exist_ok=True)

SMOKE = True        # True = 2-epoch test of the first combination only. Set False for the real batch.

# (architecture, seed). The classical-only seeds 42, 43 and 44 are run manually in the other notebook, so they are not in this list.
RUN_LIST = [("cnn_ms", 42), ("cnn_ms", 43), ("cnn_ms", 44),
            ("cnn_single", 42), ("cnn_single", 43), ("cnn_single", 44)]

config = {"epochs": 50, "batch_size": 32, "lr_backbone": 1e-4, "lr_other": 1e-4,
          "weight_decay": 0.01, "n_trans_layers": 4, "tag": "ctrl"}
if SMOKE:
    config["epochs"], config["tag"], RUN_LIST = 2, "batchsmoke", RUN_LIST[:1]
print("SMOKE" if SMOKE else "REAL BATCH", "| epochs", config["epochs"], "| tag", config["tag"], "|", RUN_LIST)

SMOKE | epochs 2 | tag batchsmoke | [('cnn_ms', 42)]


## 3. Augmentation (same as every other experiment)

In [ ]:
train_tf = A.Compose([
    A.Resize(256, 256),
    A.RandomResizedCrop(size=(224, 224), scale=(0.85, 1.0)),
    A.HorizontalFlip(),
    A.VerticalFlip(),
    A.Rotate(limit=15),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_tf = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

## 4. Data: APTOS, IDRiD and DDR (deduplicated), each cached once at 256x256 (area interpolation)

In [ ]:
def cache_one(a):
    s, d = a
    if not os.path.exists(d):
        im = cv2.imread(s)
        if im is None: raise RuntimeError("cannot read " + s)
        cv2.imwrite(d, cv2.resize(im, (256, 256), interpolation=cv2.INTER_AREA))
def cache_all(df):
    with ThreadPoolExecutor(4) as ex:
        list(tqdm(ex.map(cache_one, zip(df.src_path, df.cache)), total=len(df)))

# ---- APTOS (cleaned split) ----
APT_ROOT = "/content/APTOS"
ap_dir = {"train": f"{APT_ROOT}/train_images/train_images", "val": f"{APT_ROOT}/val_images/val_images",
          "test": f"{APT_ROOT}/test_images/test_images"}
ap = pd.read_csv(BASE_PATH + "/aptos_clean_split.csv")[["id_code", "diagnosis", "split"]].copy()
ap["src_path"] = [f"{ap_dir[s]}/{i}.png" for s, i in zip(ap.split, ap.id_code)]
ap["cache"] = [f"/content/cache_aptos/{i}.png" for i in ap.id_code]
ap["diagnosis"] = ap["diagnosis"].astype(int)
os.makedirs("/content/cache_aptos", exist_ok=True); cache_all(ap)
for sp, gdf in ap.groupby("split"): print("APTOS", sp, len(gdf), np.bincount(gdf.diagnosis, minlength=5))

# ---- IDRiD (all 516 graded images) ----
R = "/content/idrid/IDRID"
img_dirs  = {"train": f"{R}/1. Original Images/a. Training Set", "test": f"{R}/1. Original Images/b. Testing Set"}
lab_files = {"train": f"{R}/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv",
             "test":  f"{R}/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv"}
rows = []
for sub in ("train", "test"):
    lab = pd.read_csv(lab_files[sub]).iloc[:, :2].dropna(); lab.columns = ["name", "grade"]
    files = {os.path.splitext(f)[0]: os.path.join(img_dirs[sub], f) for f in os.listdir(img_dirs[sub])
             if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))}
    for _, r in lab.iterrows():
        n = str(r["name"]).strip()
        if n in files: rows.append(dict(sub=sub, name=n, src_path=files[n], diagnosis=int(r["grade"])))
idr = pd.DataFrame(rows)
idr["cache"] = [f"/content/cache_idrid/{s}_{n}.png" for s, n in zip(idr["sub"], idr["name"])]
os.makedirs("/content/cache_idrid", exist_ok=True); cache_all(idr)
print("IDRiD", len(idr), np.bincount(idr.diagnosis, minlength=5))

# ---- DDR (deduplicated; the same 12,424 images and row order as before) ----
by_name = {}
for root, _, files in os.walk("/content/ddr"):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")): by_name.setdefault(f, []).append(os.path.join(root, f))
dd = pd.read_csv(BASE_PATH + "/ddr_clean_split.csv").sort_values("csv_row")[["id_code", "diagnosis", "csv_row"]].copy()
dd["src_path"] = dd.id_code.map(lambda n: by_name[n][0])
dd["cache"] = [f"/content/cache_ddr/{i}.png" for i in dd.id_code]
dd["diagnosis"] = dd["diagnosis"].astype(int)
os.makedirs("/content/cache_ddr", exist_ok=True); cache_all(dd)
print("DDR", len(dd), np.bincount(dd.diagnosis, minlength=5))

class CachedDS(Dataset):
    def __init__(self, df, tf):
        self.p, self.y, self.t = df.cache.tolist(), df.diagnosis.astype(int).tolist(), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        im = cv2.cvtColor(cv2.imread(self.p[i]), cv2.COLOR_BGR2RGB)
        return self.t(image=im)["image"], self.y[i]

def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

bs = config["batch_size"]
mk = lambda df: DataLoader(CachedDS(df, val_tf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
val_loader, test_loader = mk(ap[ap.split == "val"]), mk(ap[ap.split == "test"])
idrid_loader, ddr_loader = mk(idr), mk(dd)

100%|██████████| 3504/3504 [05:17<00:00, 11.03it/s]


APTOS test 339 [197  28  75  12  27]
APTOS train 2824 [1427  277  756  145  219]
APTOS val 341 [172  33  91  20  25]


100%|██████████| 516/516 [00:54<00:00,  9.55it/s]


IDRiD 516 [168  25 168  93  62]


100%|██████████| 12424/12424 [02:36<00:00, 79.49it/s]

DDR 12424 [6168  630 4477  236  913]


## 5. Model variants, losses, evaluation

In [ ]:
class FlexNet(nn.Module):
    def __init__(self, arch="classical", num_classes=5, embed_dim=256, n_trans_layers=4):
        super().__init__()
        self.arch = arch
        self.backbone = timm.create_model("efficientnet_b2", pretrained=True, features_only=True)
        feat_dims = [f["num_chs"] for f in self.backbone.feature_info[-3:]]
        if arch == "classical":
            self.proj = nn.ModuleList([nn.Linear(feat_dims[i], embed_dim) for i in range(3)])
            enc_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, dim_feedforward=embed_dim * 4,
                                                   dropout=0.5, batch_first=True)
            self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_trans_layers, enable_nested_tensor=False)
            self.pool = nn.Linear(embed_dim, 1)
        elif arch == "cnn_ms":
            self.proj = nn.Linear(sum(feat_dims), embed_dim)
        elif arch == "cnn_single":
            self.proj = nn.Linear(feat_dims[-1], embed_dim)
        else:
            raise ValueError("unknown arch: " + arch)
        self.classical_head = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, num_classes))
    def forward(self, x):
        feats = self.backbone(x)[-3:]
        if self.arch == "classical":
            tokens = torch.cat([self.proj[i](f.flatten(2).transpose(1, 2)) for i, f in enumerate(feats)], dim=1)
            tokens = self.transformer(tokens)
            attn = torch.softmax(self.pool(tokens), dim=1)
            pooled = (tokens * attn).sum(dim=1)
        elif self.arch == "cnn_ms":
            pooled = self.proj(torch.cat([f.mean(dim=(2, 3)) for f in feats], dim=1))
        else:
            pooled = self.proj(feats[-1].mean(dim=(2, 3)))
        logits = self.classical_head(pooled)
        return logits, logits, None, pooled, None, None

class FocalLoss(nn.Module):
    def __init__(self, gamma=2, smoothing=0.05):
        super().__init__()
        self.gamma, self.smoothing = gamma, smoothing
    def forward(self, logits, targets, reduction="mean"):
        logits = logits.float(); K = logits.size(1)
        with torch.no_grad():
            t = torch.full_like(logits, self.smoothing / (K - 1))
            t.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        ce = -(t * F.log_softmax(logits, dim=1)).sum(dim=1)
        loss = (1 - torch.exp(-ce)) ** self.gamma * ce
        return loss if reduction == "none" else loss.mean()

def ordinal_loss(logits, targets):
    cum = torch.cumsum(torch.softmax(logits.float(), dim=1), dim=1)
    idx = torch.arange(cum.size(1), device=cum.device).unsqueeze(0)
    return F.mse_loss(cum, (idx >= targets.unsqueeze(1)).float())

def compute_total_loss(hybrid_logits, classical_logits, quantum_logits, targets, focal_fn):
    w = torch.ones_like(targets, dtype=torch.float32); w[targets >= 3] = 1.5
    total = (w * focal_fn(hybrid_logits, targets, "none")).mean()
    total = total + 0.3 * focal_fn(classical_logits, targets) + 0.1 * ordinal_loss(hybrid_logits, targets)
    if quantum_logits is not None:
        total = total + 0.2 * focal_fn(quantum_logits, targets)
    return total

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
model = None

@torch.no_grad()
def evaluate(loader):
    model.eval(); P, Y = [], []
    for imgs, labels in loader:
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(imgs.to(device, non_blocking=True))[0]
        P.append(torch.softmax(out.float(), 1).cpu().numpy()); Y.append(labels.numpy())
    P = np.concatenate(P).astype(np.float64); P = P / P.sum(1, keepdims=True)
    Y = np.concatenate(Y); pred = P.argmax(1)
    L = [0, 1, 2, 3, 4]; cm = confusion_matrix(Y, pred, labels=L)
    m = dict(acc=accuracy_score(Y, pred),
             f1=f1_score(Y, pred, average="macro", labels=L, zero_division=0),
             qwk=cohen_kappa_score(Y, pred, weights="quadratic", labels=L),
             auc=roc_auc_score(Y, P, multi_class="ovr", labels=L),
             ref_auc=roc_auc_score((Y >= 2).astype(int), P[:, 2:].sum(1)),
             sens=cm[2:, 2:].sum() / cm[2:].sum(), spec=cm[:2, :2].sum() / cm[:2].sum())
    return m, P, Y, pred
print("device:", device)

device: cuda


## 6. The batch: every combination in RUN_LIST, one after another

In [ ]:
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
names = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]
res_path = f"{BASE_PATH}/results_controls.csv"
def log_row(**row):
    pd.DataFrame([row]).to_csv(res_path, mode="a", header=not os.path.exists(res_path), index=False)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def run_one(arch, seed):
    global model
    RUN = f"{config['tag']}_{arch}_L{config['n_trans_layers']}_s{seed}"
    done_marker = f"{BASE_PATH}/ext_ddrclean_{RUN}.npz"
    if os.path.exists(done_marker):
        print("\nSKIP (already finished):", RUN); return
    t_start = time.time()
    print("\n" + "=" * 90 + f"\nRUN {RUN}\n" + "=" * 90)

    set_seed(seed)
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(CachedDS(ap[ap.split == "train"], train_tf), batch_size=bs, shuffle=True, num_workers=2,
                              pin_memory=True, generator=g, worker_init_fn=seed_worker, drop_last=True)
    model = FlexNet(arch=arch, n_trans_layers=config["n_trans_layers"]).to(device)
    criterion = FocalLoss(gamma=2, smoothing=0.05)
    bb_ids = {id(p) for p in model.backbone.parameters()}
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in bb_ids]
    optimizer = torch.optim.AdamW([{"params": list(model.backbone.parameters()), "lr": config["lr_backbone"]},
                                   {"params": other, "lr": config["lr_other"]}], weight_decay=config["weight_decay"])
    warmup = LinearLR(optimizer, start_factor=0.1, total_iters=5)
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, config["epochs"] - 5))
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])
    print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M | schedule: {config['epochs']} epochs, cosine T_max {cosine.T_max}")

    best_path = f"{BASE_PATH}/checkpoints/best_{RUN}.pth"
    last_path = f"{BASE_PATH}/checkpoints/last_{RUN}.pth"
    start_epoch, best_qwk, log = 0, -1.0, []
    if os.path.exists(last_path):
        ck = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["opt"]); scheduler.load_state_dict(ck["sched"])
        start_epoch, best_qwk, log = ck["epoch"] + 1, ck["best_qwk"], ck["log"]
        print(f"Resuming from epoch {start_epoch}, best val QWK {best_qwk:.4f}")

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    for epoch in range(start_epoch, config["epochs"]):
        t0 = time.time(); model.train(); run_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out, cl, ql = model(imgs)[:3]
                loss = compute_total_loss(out, cl, ql, labels, criterion)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            run_loss += loss.item()
        scheduler.step()
        vm, *_ = evaluate(val_loader)
        log.append(dict(epoch=epoch + 1, train_loss=run_loss / len(train_loader), minutes=(time.time() - t0) / 60,
                        **{f"val_{k}": v for k, v in vm.items()}))
        print(f"Ep {epoch+1:03d} ({log[-1]['minutes']:.1f} min) loss {log[-1]['train_loss']:.4f} | val acc {vm['acc']:.4f} "
              f"f1 {vm['f1']:.4f} QWK {vm['qwk']:.4f} AUC {vm['auc']:.4f}")
        if vm["qwk"] > best_qwk:
            best_qwk = vm["qwk"]
            torch.save({"epoch": epoch, "model": model.state_dict(), "val": vm}, best_path)
        torch.save({"epoch": epoch, "model": model.state_dict(), "opt": optimizer.state_dict(),
                    "sched": scheduler.state_dict(), "best_qwk": best_qwk, "log": log}, last_path)
    pd.DataFrame(log).to_csv(f"{BASE_PATH}/train_log_{RUN}.csv", index=False)
    print("Training finished. Best validation QWK:", round(best_qwk, 4))

    # ---- evaluation, once per dataset, with the best-validation checkpoint ----
    ck = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"]); be = ck["epoch"] + 1
    tm, P, Y, pred = evaluate(test_loader)
    print(f"\n--- APTOS test ({len(Y)}), best epoch {be} ---"); print({k: round(float(v), 4) for k, v in tm.items()})
    print(classification_report(Y, pred, target_names=names, digits=4, zero_division=0))
    print(confusion_matrix(Y, pred, labels=[0, 1, 2, 3, 4]))
    np.savez(f"{BASE_PATH}/test_preds_{RUN}.npz", P=P, Y=Y)
    log_row(run=RUN, dataset="aptos_test", best_epoch=be, n=len(Y), **{k: float(v) for k, v in tm.items()})

    tm, P, Y, pred = evaluate(idrid_loader)
    print(f"\n--- IDRiD external ({len(Y)}) ---"); print({k: round(float(v), 4) for k, v in tm.items()})
    print(confusion_matrix(Y, pred, labels=[0, 1, 2, 3, 4]))
    np.savez(f"{BASE_PATH}/ext_idrid_{RUN}.npz", P=P, Y=Y)
    log_row(run=RUN, dataset="idrid_516", best_epoch=be, n=len(Y), **{k: float(v) for k, v in tm.items()})

    tm, P, Y, pred = evaluate(ddr_loader)
    print(f"\n--- DDR deduplicated ({len(Y)}) ---"); print({k: round(float(v), 4) for k, v in tm.items()})
    print(confusion_matrix(Y, pred, labels=[0, 1, 2, 3, 4]))
    log_row(run=RUN, dataset="ddr_dedup", best_epoch=be, n=len(Y), **{k: float(v) for k, v in tm.items()})
    np.savez(done_marker, P=P, Y=Y, csv_row=dd.csv_row.values)       # written last = marks the run as finished
    print(f"\nFINISHED {RUN} in {(time.time()-t_start)/60:.1f} min")
    del model; model = None
    if device.type == "cuda": torch.cuda.empty_cache()

for arch, seed in RUN_LIST:
    run_one(arch, seed)
print("\nALL RUNS IN THE LIST ARE FINISHED")

## 7. Summary of everything finished so far

In [ ]:
r = pd.read_csv(res_path)
r = r[~r.run.str.startswith("batchsmoke") & ~r.run.str.startswith("ctrlsmoke")]
print(r[["run", "dataset", "best_epoch", "n", "acc", "f1", "qwk", "auc", "ref_auc", "sens", "spec"]].round(4).to_string(index=False))